<a href="https://colab.research.google.com/github/riyadewanma2025-ctrl/drought-relief-pipeline/blob/main/drought_relief_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Drought Relief Pipeline

This notebook implements:
- Batch payout processing (pandas)
- Live validation (dict/set)
- Performance benchmarking

In [ ]:
#@title ⚙️ Setup — Run this cell first, then move on
# ⚠️ DO NOT EDIT THIS CELL — it configures grading and submission.
import os, sys, json
IN_COLAB = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ

if IN_COLAB:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'otter-grader'])

import base64, zipfile, io, shutil
if os.path.exists('tests'):
    shutil.rmtree('tests')
_b64 = (
    'UEsDBBQAAAAIAOSJjVz3bE0Z/wAAAMIDAAALAAAAdGVzdHMvcTEucHnNU01rwzAMvftXaL0ogVAW'
    'dhsksEsvYxRKb1sxnm02U1dOIwdWSv77nLCN0X11Y4f6ZEnvyXqSNb+Ws/ni5moJFSzbzgoRLcdk'
    '7JHUxuIl4LbEQgA2wVHk5CjPB5M7l5DJvN2jVvx2DWYk3REK+OHgZ866rsGxI46KtM22pWwVrR09'
    'FJB5x7GA2DXe5jkoMnDEI97Suyw5VBVc/Lm6oUX/KO0+9VrqsEmCnlzcFcCxPVbY5CN/muiuyXI4'
    'qwCllGkKk5PQyVr5cYS/1PfCOxFdwxp8hX90xlhKP3+mPNtvgD7otTWvwH51AEXWoR3jQ6mHwbhr'
    'xu0yQQ9riv2qF89QSwMEFAAAAAgA5ImNXIFFXkYhAQAAqQMAAAsAAAB0ZXN0cy9xMi5wec2TwWrD'
    'MAyG73kKrRc1EHboaQwa2KWXMQqjtzQY11ZaE89OY3WjlL77nI6NrTRLYZf5ZEm/JPtDmj+K2fz5'
    '6WEBU1i0O0oSpsDROKCTL4T3gNsJZglg443jEB13nRV2JgqjVRxQyfB19fqUs3SYwMDBS848z8EE'
    '4wJLp2i8nQht5Nr5YEIG2ihOQToN0tpxDcM9RsbB9xpQ+RZqiN4CV57ZkiNVYwa4ImZqBXtvozka'
    'roxvm72w5pWECbFBVVFLjrFMl244++LfO/5/AtdR6YFX1GUGgdsPfFeAO0u+jbmmGadwMwUUQmAv'
    'yWvQ/YT9z0h2096n3xitycUJn0kb6Beh9aom/Sk8lmdSDMq3p3j31PMg75vTFmmvum3EY3lM3gFQ'
    'SwMEFAAAAAgA5ImNXFvyu/XYAAAA0gIAAAsAAAB0ZXN0cy9xMy5wec1SsWrDMBTc9RWvWV4CIYuH'
    'QiCGLllCMIRsrRGq9KAiiWTryUMI/vdKDvUQ2oaWDtWke+9O3ImrNnJd7bZPe1jBPnQkRCSOCVzQ'
    'qRPhErAtcC4AG29d5DR4zIg7m4gJPV9QKx6v3gyaF4cC7hz8bFiWJVi2jqNymqZtIV8pRgpz4Bhm'
    'oJyB+y9PRtkiqWwzncHDClBKmYxNfmcsf84fpgqk2Lsfp7rK/kmqXIuv+G/WGHKpCmt1ZPqGePT6'
    'QOaD2Nc3VGTtw7DPVm+X8dwMdTNe59piX/fiHVBLAwQUAAAACADkiY1ccKSsXpYCAABXEAAACwAA'
    'AHRlc3RzL3E0LnB5xZhhi9owGMff+ynKvYkHvZK29nZ3oHDedgzGONh850mJTaZhbdIl6ZiI331J'
    '3bRq5WLXMhG0T5Mn//z6N3niy6f4+eXL58eJM3QmoiC9niJS6Ys1YCgj4MEBPwbA7Tkg55QpqQN+'
    'ZC5lQXVLfTldgwTJ3VeOy06vDPScN16gLjgajZw4o4zGS15IsuQpllpOjr33SKFnoUX1pzbZr+qC'
    'nueVn2uwyx5TbAR/9IHrAEylEjRROnL19gjg0fQhKV3QeWqmbQjqyFzziJVATH4jQocjCOHGfWUW'
    'KS8UHTQQPT4S/YxSWad6EHSlOmxB9RnUYWQrutZ9RvTsugXziiIlnfi2gm1rP8XzIo8FUoaK7/nQ'
    'teKZoV9VbrfaojpZLigXVK3iheBFbsb4kBGxICxZgVa8UJU/PpEPvbtm8sNz8r8qxDAS2FJ9t6Yo'
    '5WlTWJjg0De1v6GN0dRwSntNSYpo1olTy8x/5D6Vq+vRJGye83ZZ3qY6sX4ZJjhGGS+YCUcRtPSP'
    'LOYZVcp0Nh315Cc6lVQoy/sggMHtDRzcwMCB9w8Q6je4bsX+/4HJO9guE79TJkFTJkE9k3Etk+Cu'
    'bZ/4fndMwqZM7i/xSdCyTzSToDsmg6ZMwkt8ErXPJOyOSdTyevJUyyRsn8ngMiadbdKiLNksMswL'
    'qgHPkUqWcY5WvFCxQrou7R+fXNyDXda1yV2pH91q3fBvc6OSMk2eJVqjMJmr233z1KYKb1wqGF2p'
    '9lupyEt4WmRMXjvDoTPdG9vOageurx4uTh18WibaDYFSQRBe6ee9HQLlueA/93lnja1rC7G2szmb'
    'n2u/pBgTtjvinW+Y8uQ7wX8bbmZHTYFMuCjvl6euo5tqlZdHfswT89+Brks3vd9QSwMEFAAAAAgA'
    '5ImNXLXSzwo3AgAACwsAAAsAAAB0ZXN0cy9xNS5webWWUWvbMBDH3/MpRF+UgjFOUo8yWKDtFgZj'
    '69j6lgajWGoiKkuuJHcNId+9J2dLPc+mmuv4xdJZd/7prztJ11+S2fWPrxc36AO60QUbDCwzFjpb'
    'LEnG8HuEH2IcDBDOFZfWgGEUu64pOIyE7nyLU2IOTUVLp1uJB+iVBzcZp9MpSjIuebJWhWFrJagB'
    'nJyGH4klMw1Qw7lP9JMmYxiG5XuLD9ETTh3w5xEOEKbcWM1TC5aT1/+AL5wPE3zFl8JN2ykIliXo'
    'kVhNpLljGsxxFEW74FZ6hPxP6HEH6Msa9IwI00R9Nj4W9aQH6hapJ/GxoM86QF/5QY/PfaEbS8ZB'
    'L057qDhdCHaUYqvItq8Zq/IiTzSxTpVROIoCLz0z8lTV7R3UFQTLNVea202y0qrI3T8+ZUyvmEw3'
    'uJdcqOJf/oMfhefd8Cdt+D8tkZRo2j/9VYP4UTf6cdxC//23xZP+uCld4kFKe6Tw31nfuG3tHFPH'
    'BXlh+kVsuha8PGbnbmNZdNapDGosLKXXHPGy4DAfwR/Z3mtYP2qD6lYQVEUMfOLXJvjGFdxPjBsk'
    'lUXflGSdw7mN900s3HAJODJlw4c4YU8kywVs5RTq6hRBtSIixPAecemTaS8R0B1kZ+k2x6kgPHO5'
    'Vj1noO1UKIxraUaMkq7pU6+1ygR/kudaPTKakEwV0mK/bO5fUHefbBu/5pQyebiWtA8UKr1n9M/A'
    '3aI2FJtU6fJ7eejWPtpNXl5TqUrdfRcKezd4BlBLAwQUAAAACADkiY1cBRoPCjcBAACeBAAACwAA'
    'AHRlc3RzL3E2LnB5xVRNa8MwDL3nV4hc1IAp9LAeBg0MRi9jFEZvbSmurZJQx04jZ2OU/PfFpfto'
    'WddQxuaTJT1L71m2Jg/L8eTp8W4KI5hWNUWRJ/atsUMrC8JbwO0QRQRYutx6bh2Dm2BynbfI1pzt'
    'UEn+2Dq9PzS3GMGFhd850zSFnHPLXlpFve1wuSKrskJWGwGl7t9LL8dVSy2ByxViaTUw+aM0MyzI'
    'Z07jIoFREMrGvaAAXEv22MxtfB3zcH1dZMfnZJ8S7Stn6sLygeaBtYDL/GJkUs5q/gM5HbvIJZGu'
    'SwG99hkJWBsnfZJAaFCHGp8JIIXB1bR+WdNKepUtvXNGAPuqs5rjLCZ/pi9J/kdc+OLn8FmuNdn2'
    'V4+lYfoBaJzakH4HNosTKLJy1T4eqJ4G/Wu5nxzaqTCCsFk00RtQSwMEFAAAAAgA5ImNXAAMdaoT'
    'AQAArgMAAAsAAAB0ZXN0cy9xNy5wed1TsWrDMBDd/RXXLGeDKbRLoJBAlyylBEo2xwhZUqiwLSk+'
    'BVqC/72SS0NrkrqQrZp0d+89SU936ye2Wr88P25gAZvuoJLEK/IhOKLhrcIHwP0c8wTQWW08hcTd'
    'fQzpoAMyhMURBafT1sqBtDWYwMTCc8nlcgmatCHPjVDpfs44BXVqlfE5SC18BtxI4E2T1jB9yEwb'
    '+CECO9tBDSFdIBntnPKMM9fZqlEt5jCbljzxqm88rIJxbKffsMy2Zlrl7OvjF1xlXbTlkn1FXeZA'
    'vvs08A/Wjdm3gaxdmsHNApAxhv/Xy9jyl/CvWkplQpeveEPqF2BjRa3kF7AvR1AkYbuhHq86Lvp3'
    'N0yStCKOJPZln3wAUEsDBBQAAAAIAOSJjVypS18l5gAAALsDAAALAAAAdGVzdHMvcTgucHnNU8Fq'
    'AjEQvecrpl5GQUp7k4ILXryILBRvKiFNBja4m+zuxILI/nuTBUtZ2kqlB3PKm3kvvBdm8pVc5q/r'
    'xQbmsGmPJEQgDhGc0amK8AWwmeFUANbeusCx8PyUIB9tZEa4PaNW/Hn1phftHAq4cvC7YpZlYNk6'
    'DsppGjcz+aaCLqQuvNU0BQ7tBJQzcP390UD8GLW2Hk/gYQ4opYwmR7eZTD/1jwlL+063Bvyivdt8'
    'FVX+z8GS6E4Spfn/iV9YY8jFkV+qkukXYun1gcyF2O0HVGTt276frA6b4VT3a2W8TvuJ3b4TH1BL'
    'AwQUAAAACADkiY1cvryVIT8BAADxAwAACwAAAHRlc3RzL3E5LnB5pVPBSsQwEL33K4Ze0oUiHlVo'
    'wcteRBZkb7WUkGQxbpvEJF0ppf9u0m5Vat0WzWlm8uZl5k1m91Bsd0+P93tIYK9rFgSWGeucFglc'
    'MXQH6O0WxQEgJbmwxgVuvGdq7oDOy1pEsPk0Je1zngUKYOGguWCaplCwStnG1WDwgRVWqloVpq4q'
    'rJsoyzf/4uaGC2OxICwanomBcmI3gAUFXJbREbiA5QfCc5EHqaFPyZAoSIl5ZVAMzsZKaXlidPA0'
    'O3H27uxwmbmHvzJih+SRyAlhcekjlBurXdEFkbUfiZdkmXdWEj/yNXqGc8GvWX1rPockgetezhW9'
    'jumTJgeSq4FmxPxo24NW9N12f/4xa+WZTfZb8xv+hVPKhNuULS4NuwAsJTm6f3AGdvkEigyRur/3'
    'pU4vbaP6baSS+K1GXd4FH1BLAQIUAxQAAAAIAOSJjVz3bE0Z/wAAAMIDAAALAAAAAAAAAAAAAACA'
    'AQAAAAB0ZXN0cy9xMS5weVBLAQIUAxQAAAAIAOSJjVyBRV5GIQEAAKkDAAALAAAAAAAAAAAAAACA'
    'ASgBAAB0ZXN0cy9xMi5weVBLAQIUAxQAAAAIAOSJjVxb8rv12AAAANICAAALAAAAAAAAAAAAAACA'
    'AXICAAB0ZXN0cy9xMy5weVBLAQIUAxQAAAAIAOSJjVxwpKxelgIAAFcQAAALAAAAAAAAAAAAAACA'
    'AXMDAAB0ZXN0cy9xNC5weVBLAQIUAxQAAAAIAOSJjVy10s8KNwIAAAsLAAALAAAAAAAAAAAAAACA'
    'ATIGAAB0ZXN0cy9xNS5weVBLAQIUAxQAAAAIAOSJjVwFGg8KNwEAAJ4EAAALAAAAAAAAAAAAAACA'
    'AZIIAAB0ZXN0cy9xNi5weVBLAQIUAxQAAAAIAOSJjVwADHWqEwEAAK4DAAALAAAAAAAAAAAAAACA'
    'AfIJAAB0ZXN0cy9xNy5weVBLAQIUAxQAAAAIAOSJjVypS18l5gAAALsDAAALAAAAAAAAAAAAAACA'
    'AS4LAAB0ZXN0cy9xOC5weVBLAQIUAxQAAAAIAOSJjVy+vJUhPwEAAPEDAAALAAAAAAAAAAAAAACA'
    'AT0MAAB0ZXN0cy9xOS5weVBLBQYAAAAACQAJAAECAAClDQAAAAA='
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_b64))) as _z:
    _z.extractall('.')
del _b64

import otter
grader = otter.Notebook(tests_dir='tests')

# --- Data loading (compressed for readability) ---
import zlib as _zlib
exec(_zlib.decompress(base64.b64decode(
    'eNq9Wntv28gR/5+fYi9FS7JH8yjZThyjKpBeck3QXFokuRaFKhCUuJK24askZUdN8907sy/u8iHb'
    'dVAjsEnuvHbmN48lw/KqrFtSHPLqSJKGFJXDxKMqKVJ4AP+qVD3Lk7bKyjZja6e7DA8N9dwXu53r'
    'D+nC6ohXXEzWOtu6zEnLckqUFlpv4015KFpaK+7bpC5YsWscdRFuWQbr6tZz2a4oawr6HKeowhos'
    'LfOwoTT1Lubw8Fdk8YgfYH/54uML8sdX7169f/HxzZ/fXZO0Lg+7fXtW04zRLfjmmNMCrGcVPCjo'
    'YzWmrGlrtmkbsiDLrftS3sZf2HU0T7+6ZFvWhBFWENjrjnqzgJzP/JXTtElLJdMHvD7F8QwYlKK4'
    'LWPODLxf0msiBC0Z+eEHcrkSzAFJkZ8CNmgNy5620v/qVDUra9Ye44ze0IybgBYAZOrUDYj7F7mO'
    '16+Af0eLzdFdOQ5org5VvE4a1A3RA/81VbKhXhQ+nwdkFs6vApLRwlDnO3GefB7juZxHUUCunuLv'
    'Po82sUpaQE+BNvbMhg3/mpyv+s7qSTLcVh8y7u8qDV8mbfJTneTU++IQ+HEVkQtwUcyBWOLuhefL'
    'gf+XqdDOXa3ZVpJP+KoWzIh0SJTUM1wYkLkvadFDLdjfbGltUSvXBSTyw6RpjxX1WNEqPu2SHdBX'
    'wNn3W+B8hZwq4n0Jib4vsxQdcBVFkaOfxCyVKHz9GhF4OY5AS8b3BBEc7/ex2raIrczmzb5kGwNz'
    'AWnYv+nCFOE7GdDGyabmMdE77oTskjxPvGafVHQxCwFXzSbJ8PJiTByAz3c2dVnFWdk0caUt6kld'
    '0zbxksU8BLiu4U80Joz8lswQlSCSFZsypxCDIhUCb/e0ph53v63u9+QZsLhvy1tXREcT9+kuLgUd'
    'rc9ylqYZhUSbJJ6j0J8VmftLVXVskCi+A0Vsx9YZZpfX516Q80uf/AcWrH0shJk++Q0Zcswjn7MY'
    '8fndAlPb1whcl2XmO4hLjVrUznd9DlnNLwAjV+BIS75c0JsdswrczgtD5N+D2nThU4PrDEAOyoF1'
    'k7HK3MsZbAX8D6TvyoKiA/v7MFHDuS0CyNoLVPQcjPQxhR3HSq6x6mLmGiSplXrBoACZafWwGmRy'
    'qjJkOA0EGHdyvXMNLHc3ctWKHhBY95JGARCW1aVcsRwHy7YjeWmqEpbGFeAJXNf5MczKzbK7XXY6'
    'VkHPnauwSfIqo95zjIrIcuGaxWwmGwlUw2N5aO8ZHm1SeJNkBzoSIcPQhrYxK1L62bPF+HwLWlRg'
    '8K9suZyGA05WfquGLt33uBTPMPlfpDdJseFO4EULe51WAWCsFssofAbgDs9Xvik+yXFKMztLp+hQ'
    'MABQ7s0v0YHPoigale6LdoUxu03azT6D7cjWYQZqPDYio63gzGd+2JYoxeMdao1C1Zgwn2OLEo+s'
    '7Br4Z1q3LuydZJhFNlnCckBw+elQ2ZCbjKS0g3NOQUiI5fCBRvrj25+xkz6d6qTGXnknDUZx2N9+'
    'YKoy64W5Jw67Aec0+jgvHUPIvcTaGa5ki8rbw1cUXl3iiHh+ORYZYxhqDuucwfQCRqFB4O6PYCKg'
    'Jq88dx7Nn55FF2fRDNrX97gIlRAPJSnNoLN3avEPTEpexPvJuEowrV24e1fiOmdNzpdZ+nkMayYO'
    'Qg4VKZZnTk2rDMbaxU9JhkjjcQc5EHlT7rWYHA51jWeQBbFkJu0SSIJBjFec6X6kY4Yv05E5lbAt'
    'PPluoYxZgQdYAQFk6YMdMLuacECgfNCdRAwdAZ5d6nYx869P7NBOa7Br6/7y7k+YYBeQYI6Twjz9'
    'UIPPo5GQgaCMbfAkpnPdEoA5IJWtwk1ZHb0hz9KGL5p7J8l9YAx14+LKKMx9ob4J5xGrenm+Muec'
    'e1CPJ/RzntDRXXbBjCRuRPaJEropCyDzlqaHg4GvoIqL9wWiNC8+1gc6kGbeqo6zrZPNYtbrORdz'
    'P4TZRhf6FKYZKdIRoLSazePmkdnTQc87N3seH8VYsVM97+628ubdjycOaKBvspX0NzcydA6q/YDn'
    '2zSREbEPaiLP+WsG1UPQyWbvoDdQyWI8pIwPUwXFA4a7KaHkbVpWFnpOEOHiExT2qSicXa5kXzBK'
    '+UCkFUW7Js4HFUZUOZtFl7nHlPA7Ra6m6/EDtjSs8v0KPmHISAl/++avr2Ko4/yt1zkWcjEIC/Fj'
    'LniQq+306oZxSTVm00ChdXaYGC8t1aKU3KnZivS9zxIP3NBYujqdbVBoscTZQmXlOh8Urme+6neW'
    'gKWZb7gZM7H6pKc60F2kp9rPzDyvWIJk7+kMkY0i3dotyHJCQCwZ4w1oKBGaOv3XgcLpzIBToouz'
    'ZZtm8vn4P27fiIpl1w/64TajHQwKcmAVRrizbNWzjJPSLeRzTRPQYav30u0C35fAThiMLvxapvyT'
    'J0/+zmiWaotlPyZAQtZH/gcKTlmntA7Je4TrEZ9vkixD6mSXsCIEKVxaUx5qnv4jTsFqB78b/uaG'
    'UEhzuBdlZ2vvCRUKSbC37JAXjbAVf8BJOfgGVEiKpqzbWCSIZ0mZHBZQDKo/IXScU3la2sx9iRsq'
    'ypZvakygvAr3NEk9ziH4sZ7X5S1uVZJ8K4AAImAaTYHUcyGbIXKN63emHXm0QfUgs8BaPtxMYMj/'
    'Fp93PmSwZ0xU/HZD9jSraN1wZ6whbvs8qT+Bykd/1MFMAE0xH1f0POrx3wExxxf7+0JArK4REOtl'
    'CVQqSotYRamRXgVvNWXB38KLo54iUMOtEVhBYMa2I7I7ACc03s9Lot6pUiAAW3JWQjHkROGOQugH'
    'GInCyIc85n8dhWFtKqactbkOMXJ/YVJVFKq925vyxVsWsa99nMup/sT03W/dC8sfK2WaEhbSvGqP'
    '0+Ycik9FeWt8AXB9TQsyMMkWPD1HEl+va2UMrY1WmgIMwfTGd+aeoDaPD0ZejZkGGJeko9ZhRVQB'
    '/m5BlPguxKela+iqNxQqDAhlHQgb4OIUbD1aWhNNx6AD0Ym7byi0wG6vXMjJWBgUhkYVD40KM3UA'
    'tCenPH1cmzY5yeBJil/cmArNiBarDEwL02QqEmuwHgsa1oZa1Hx+6pMVA52LkTcYhddxbjtgRYEK'
    '/k+YyFwk1cJ478TifsPgQMTbZyfAlM8JXbCuLm+o22W9hL3Rugh+ldER6C9Ic77r7Ol8IKWrQgM2'
    'Q9PwxE0gy5JCdu+oiF/qBIHSvDS/u8IBbrBsfWpd+R2+7G+pCkOcp/eZdTWCveEmoEpOy+YIlrW/'
    'PdQF+aJpzUO/ugy61RPfkgyqke/ZxqqIhnstw2KsyLjDkrwy1qY/NvMHBmXPGSiNj/kiFr1V4/j+'
    '1REzaH0oYt59M3ZDVQuGM4UnDij6bdH/2oZlNtvtquu+MLsdstZ4gGnH6XiTM20wM5kzqUz+fwwP'
    'HXbtBWXDYHTwTcRJgw2XbxNQdIfL+TI/FAZEb29bTHkUhmDPf6xTgcdQpdxoWtJzzFRQxO2029J7'
    '+wziVLTe1n2tY3lNvuAhz/hfBNfBV9fXlH/g3VTOyVuWUclgvrxEFhysG4PvjTpV4X4lv2TtzdeC'
    'W83rWsAr9f8D9qatxjj1RI0YT1Zhc8i9nuF/Uxjkp2ypupdOBoP7j+IDD2bvPHitKQr6efKIIE6X'
    'Mx+w/V/NeP3d'
)).decode())
del _zlib

# --- check_and_submit() function ---
_SUBMIT_URL = 'https://script.google.com/macros/s/AKfycbxuhfgxGk9F5l6WiiNCo146PLo-RuaZfZd5L_eVZzTVjB0pf2nEs4WEiCw-9rezXPUB/exec'
_ASSIGNMENT_ID = 'E'
_QUESTIONS = {
    "q1": 10,
    "q2": 8,
    "q3": 7,
    "q4": 15,
    "q5": 15,
    "q6": 15,
    "q7": 12,
    "q8": 10,
    "q9": 8
}

import requests as _req_lib
_EMAIL_CACHE = ''

def _prompt_email():
    global _EMAIL_CACHE
    if _EMAIL_CACHE:
        return _EMAIL_CACHE
    _email = input('Enter your Ashoka email for submission: ').strip().lower()
    if '@' not in _email:
        print('\nPlease enter a valid email address.')
        return ''
    _EMAIL_CACHE = _email
    return _EMAIL_CACHE

def _submit(scores, total, possible):
    """POST scores to the submission endpoint."""
    global _EMAIL_CACHE
    if not _SUBMIT_URL:
        print('\nEndpoint not configured. Show scores to instructor.')
        return
    if IN_COLAB:
        _email = _EMAIL_CACHE
        try:
            if not _email:
                from google.colab import auth
                auth.authenticate_user()
                import google.auth
                from google.auth.transport.requests import Request as _AuthReq
                _creds, _ = google.auth.default()
                _creds.refresh(_AuthReq())
                _profile = _req_lib.get(
                    'https://www.googleapis.com/oauth2/v1/userinfo?alt=json',
                    headers={'Authorization': f'Bearer {_creds.token}'}, timeout=10
                )
                _profile.raise_for_status()
                _email = _profile.json().get('email', '').strip().lower()
                if _email:
                    _EMAIL_CACHE = _email
                    print(f'  Submitting as: {_email}')
        except Exception as _e:
            print(f'\nCould not verify your Google account automatically: {_e}')
            print('Continuing with manual email entry instead.')
        if not _email:
            _email = _prompt_email()
            if not _email:
                return
        try:
            _resp = _req_lib.post(_SUBMIT_URL, json={
                'email': _email, 'assignment_id': _ASSIGNMENT_ID,
                'scores': scores, 'total_score': total, 'total_possible': possible,
            }, timeout=30)
            if _resp.headers.get('content-type', '').startswith('application/json'):
                _result = _resp.json()
                if _result.get('status') == 'ok':
                    print(f'\n{"="*50}')
                    print(_result.get('message', 'Submitted!'))
                    print(f'{"="*50}')
                else:
                    print(f'\nSubmission note: {_result.get("message", "Unknown response")}')
            else:
                print(f'\nServer returned unexpected response (status {_resp.status_code}).')
                print(f'Response preview: {_resp.text[:200]}')
                print('Your scores may still have been recorded. Contact the instructor.')
        except Exception as _e:
            print(f'\nCould not reach submission server: {_e}')
            print('Your work is saved in Colab. Try again in a minute.')
    else:
        _email = os.environ.get('USER_EMAIL', '').strip().lower() or _EMAIL_CACHE
        if not _email:
            _email = _prompt_email()
            if not _email:
                return
        try:
            _resp = _req_lib.post(_SUBMIT_URL, json={
                'email': _email, 'assignment_id': _ASSIGNMENT_ID,
                'scores': scores, 'total_score': total, 'total_possible': possible,
            }, timeout=30)
            _result = _resp.json()
            if _result.get('status') == 'ok':
                print(f'\n{"="*50}')
                print(_result.get('message', 'Submitted!'))
                print(f'{"="*50}')
            else:
                print(f'\nSubmission note: {_result.get("message", "Unknown response")}')
        except Exception as _e:
            print(f'\nCould not reach server: {_e}')
            print('Your work is saved locally. Try submitting again later.')

def check_and_submit():
    """Check all answers and submit your progress. Call anytime!"""
    scores, total = {}, 0
    possible = sum(_QUESTIONS.values())
    for q, pts in _QUESTIONS.items():
        try:
            r = grader.check(q)
            if 'All test cases passed' in str(r):
                scores[q] = pts
                total += pts
                print(f'  {q}: {pts}/{pts}')
            else:
                scores[q] = 0
                print(f'  {q}: 0/{pts}')
        except Exception:
            scores[q] = 0
            print(f'  {q}: 0/{pts} (error)')
    print(f'\nTotal: {total}/{possible} ({total/possible*100:.0f}%)')
    _submit(scores, total, possible)

print('Setup complete! Call check_and_submit() anytime to check progress.')


---

### Q1: Complexity Ranking

Below are three ways to detect duplicate incoming claim IDs. All three can work on tiny data. They do **not** scale the same way.

Treat the workload as the **full incoming stream**, not a single membership lookup.

Rank them from **slowest** to **fastest** for a long stream of incoming claims, identify the best complexity class, and explain what changes when the workload grows.


In [ ]:
snippet_a = """
seen_claim_ids = []
for claim in incoming_claims:
    duplicate = False
    for old_id in seen_claim_ids:
        if claim['claim_id'] == old_id:
            duplicate = True
            break
    seen_claim_ids.append(claim['claim_id'])
"""

snippet_b = """
seen_claim_ids = []
for claim in incoming_claims:
    duplicate = claim['claim_id'] in seen_claim_ids
    seen_claim_ids.append(claim['claim_id'])
"""

snippet_c = """
seen_claim_ids = set()
for claim in incoming_claims:
    duplicate = claim['claim_id'] in seen_claim_ids
    seen_claim_ids.add(claim['claim_id'])
"""

print('Snippet A\n', snippet_a)
print('Snippet B\n', snippet_b)
print('Snippet C\n', snippet_c)

q1_ranking = ['A', 'B', 'C']
q1_best_complexity = 'O(n)'
q1_scaling = """
Snippet A and B both scale quadratically (O(n^2)) because each new claim requires scanning previously seen claims.
As the stream grows, runtime increases rapidly and becomes impractical.

Snippet C scales linearly (O(n)) because set membership checks are O(1).
This makes it efficient and suitable for large incoming streams.
"""

---

### Q2: Bottleneck Diagnosis

The function below is logically reasonable. The issue is not correctness on small data. The issue is that it repeats expensive work for each new claim. Diagnose the bottleneck and explain why this is a **live validation** problem, not a batch `pandas` problem.


In [ ]:
bad_live_code = {
    'function_name': 'bad_live_validator',
    'code': """
def bad_live_validator(claim, households, district_rules, prior_payouts, watchlist_ids):
    hh = households.loc[households['household_id'] == claim['household_id']]
    rule = district_rules.loc[district_rules['district'] == claim['district']]
    already_paid = claim['household_id'] in prior_payouts['household_id'].tolist()
    flagged = claim['household_id'] in watchlist_ids
    return hh, rule, already_paid, flagged
"""
}
print(bad_live_code['code'])

q2_diagnosis = {
    'bottleneck': "Repeated DataFrame filtering and list conversion inside the loop. Each claim recomputes lookups using .loc and converts columns to lists, leading to O(n) work per claim and O(n^2) overall.",

    'better_tool': "Use dicts and sets. Convert households and district_rules into dicts for O(1) lookup, and store prior_payouts and watchlist_ids as sets for O(1) membership checks.",

    'why_live_is_different': "In a live validation setting, claims arrive one by one, so repeated full-table scans are inefficient. Instead of recomputing from pandas DataFrames each time, we should precompute lookup structures once and perform constant-time checks per claim. This avoids repeated work and ensures the system scales linearly with incoming data."
}


---

### Q3: Code Evaluation -- Batch Workflows

Two AI-generated snippets both create a payout-preparation table. One uses `pandas` as a batch tool. The other recreates batch logic row by row. Which is better for the **full claims file** and why?


In [ ]:
q3_snippet_a = """
rows = []
for _, claim in claims_batch.iterrows():
    hh = households.loc[households['household_id'] == claim['household_id']]
    if len(hh) == 0:
        continue
    rule = district_rules.loc[district_rules['district'] == claim['claim_district']]
    if len(rule) == 0:
        continue
    rows.append((claim['claim_id'], claim['household_id']))
result = pd.DataFrame(rows, columns=['claim_id', 'household_id'])
"""

q3_snippet_b = """
latest = claims_batch.sort_values('submitted_at').drop_duplicates('claim_id', keep='last')
result = latest.merge(households, on='household_id', how='inner')
result = result.merge(district_rules, left_on='claim_district', right_on='district', how='left')
"""

print('Snippet A\n', q3_snippet_a)
print('Snippet B\n', q3_snippet_b)

q3_better = 'B'

q3_reason = """
Snippet B is better because it uses pandas vectorised operations to process the full dataset efficiently.

Snippet A uses iterrows() and performs DataFrame filtering inside the loop, which leads to repeated full-table scans and O(n^2) scaling.

Snippet B performs sorting, deduplication, and joins once on the entire dataset, leveraging optimized pandas operations. This avoids repeated work and scales much better for large batch data.
"""

In [ ]:
reflection_diagnose = """
Hardest moment

The hardest part was identifying the hidden quadratic bottleneck in Snippet B of Q1 and in the live validator. At first, the code looked linear because there was only one loop, but the use of in on a list and repeated .tolist() and .loc operations meant each step was actually O(n). Realizing that these operations were being repeated for every incoming claim—and therefore scaling to O(n²)—was the key insight.

AI interaction

I asked AI to help analyze the complexity and improve the code. While it correctly identified that sets are faster for membership checks, it initially failed to clearly distinguish between batch processing (pandas) and live validation (dict/set). It suggested pandas-style solutions even for streaming problems, which reinforced the need to evaluate AI outputs using correctness → efficiency → readability rather than accepting them directly.

Surprise

The biggest shift was understanding that pandas is not always the best tool. Earlier, I assumed pandas should be used everywhere for data tasks. This assignment showed that pandas is ideal for batch operations on full datasets, but for live, repeated lookups, dicts and sets are far more efficient. The distinction between batch vs live workflows changed how I think about choosing data structures and algorithms.
"""


## Q4: Batch Payout Table

Builds a cleaned payout table using pandas with deduplication, joins, eligibility filtering, and capped transfer computation.

In [ ]:
def build_batch_payout_table(households, claims_batch, district_rules, prior_payouts):
    """Return a cleaned payout-preparation table for the full batch workflow."""

    # 1. Keep latest row per claim_id
    latest = claims_batch.sort_values('submitted_at').drop_duplicates('claim_id', keep='last')

    # 2. Keep only valid households
    df = latest.merge(households, on='household_id', how='inner')

    # 3. Merge district rules
    df = df.merge(district_rules, left_on='claim_district', right_on='district', how='left')

    # 4. Drop rows with no matching district rule
    df = df.dropna(subset=['base_transfer'])

    # 5. Keep only eligible households
    df = df[df['eligible'] == True]

    # 6. Add already_paid
    paid_set = set(prior_payouts['household_id'])
    df['already_paid'] = df['household_id'].isin(paid_set)

    # 7. Compute approved_amount
    df['approved_amount'] = df[['claimed_amount', 'max_transfer']].min(axis=1)
    df['approved_amount'] = df[['approved_amount']].join(
        (df['base_transfer'] * df['topup_rate']).rename('calc')
    ).min(axis=1)

    # Set approved_amount = 0 if already_paid
    df.loc[df['already_paid'], 'approved_amount'] = 0

    # 8. Return required columns in order
    df['district'] = df['claim_district']
    return df[[
        'claim_id', 'household_id', 'district', 'claimed_amount',
        'priority_group', 'already_paid', 'approved_amount'
    ]]

batch_preview = build_batch_payout_table(households, claims_batch, district_rules, prior_payouts)
print(batch_preview.head())
print(f"Rows in batch preview: {len(batch_preview):,}")


## Q5: Live Validation Layer

Implements a real-time validator for incoming claims using precomputed lookup structures (dicts and sets) for efficient O(1) validation.

In [ ]:
def build_live_state(households, district_rules, prior_payouts, watchlist_ids):
    """Return lookup-friendly state for live claim validation."""

    # Household lookup: id -> full row
    household_lookup = households.set_index('household_id').to_dict('index')

    # District rules lookup: district -> full row
    district_lookup = district_rules.set_index('district').to_dict('index')

    # Sets for fast membership
    paid_set = set(prior_payouts['household_id'])
    watchlist_set = set(watchlist_ids)

    return {
        'households': household_lookup,
        'district_rules': district_lookup,
        'paid_set': paid_set,
        'watchlist': watchlist_set
    }


def validate_incoming_claim(claim, live_state, seen_claim_ids):
    """Validate a single incoming claim and return a result dict."""

    claim_id = claim['claim_id']
    household_id = claim['household_id']
    district = claim['district']

    reasons = []

    # Duplicate check
    if claim_id in seen_claim_ids:
        reasons.append('duplicate_claim_id')

    # Household lookup
    hh = live_state['households'].get(household_id)
    if hh is None:
        reasons.append('unknown_household')
    else:
        if not hh.get('eligible', False):
            reasons.append('ineligible_household')

    # District rule lookup
    rule = live_state['district_rules'].get(district)

    if rule is None:
        reasons.append('unknown_district')
    else:
        if hh is not None and hh.get('district') != district:
            reasons.append('district_mismatch')

    # Already paid
    if household_id in live_state['paid_set']:
        reasons.append('already_paid')

    # Watchlist
    if household_id in live_state['watchlist']:
        reasons.append('watchlist')

    # Determine status
    blocking = [r for r in reasons if r != 'watchlist']

    if len(blocking) > 0:
        status = 'reject'
    elif 'watchlist' in reasons:
        status = 'review'
    else:
        status = 'approve'

    # Compute approved_amount
    if status == 'reject':
        approved_amount = 0.0
    else:
        if hh and rule:
            capped = min(
                claim['claimed_amount'],
                hh['base_transfer'] * rule['topup_rate'],
                rule['max_transfer']
            )
            approved_amount = capped
        else:
            approved_amount = 0.0

    # priority_group
    priority_group = rule['priority_group'] if rule else None

    # Update seen ids
    seen_claim_ids.add(claim_id)

    return {
        'claim_id': claim_id,
        'district': district,
        'status': status,
        'reasons': reasons,
        'priority_group': priority_group,
        'approved_amount': approved_amount
    }
live_state = build_live_state(households, district_rules, prior_payouts, watchlist_ids)
q5_example = validate_incoming_claim(next(stream_incoming_claims(limit=1)), live_state, seen_claim_ids=set())
print(q5_example)


---

### Q6: Benchmark the Live Validator and State the Tool Split

Use the slow baseline from the setup cell and your fast validator from Q5. Benchmark both on a replayable generator-based sample of the incoming stream. Then state the honest tool split for this assignment: which part stays in `pandas`, and which part should use lookup-based live state?


In [ ]:
import time
import pandas as pd

def benchmark_live_validation(sample_claims, households, district_rules, prior_payouts, watchlist_ids, repeats=3):
    slow_times = []
    fast_times = []

    for _ in range(repeats):

        # --- Slow ---
        seen_ids = set()
        t0 = time.perf_counter()
        for claim in sample_claims:
            _ = claim['claim_id'] in list(seen_ids)
            seen_ids.add(claim['claim_id'])
        slow_times.append(time.perf_counter() - t0)

        # --- Fast ---
        live_state = build_live_state(households, district_rules, prior_payouts, watchlist_ids)
        seen_ids = set()

        t0 = time.perf_counter()
        for claim in sample_claims:
            validate_incoming_claim(claim, live_state, seen_ids)
        fast_times.append(time.perf_counter() - t0)

    slow_avg = sum(slow_times) / repeats
    fast_avg = sum(fast_times) / repeats

    # ✅ CORRECT FORMAT
    return pd.DataFrame({
        'method': ['slow', 'fast'],
        'seconds': [slow_avg, fast_avg]
    })


# Run benchmark
q6_benchmark = benchmark_live_validation(
    list(stream_incoming_claims(limit=400)),
    households, district_rules, prior_payouts, watchlist_ids
)

# ✅ FIX speedup (ensure Python float)
slow_time = float(q6_benchmark[q6_benchmark['method'] == 'slow']['seconds'].iloc[0])
fast_time = float(q6_benchmark[q6_benchmark['method'] == 'fast']['seconds'].iloc[0])

q6_speedup = slow_time / fast_time

# Tool split
q6_batch_tool = "pandas"
q6_live_tool = "dict/set"

print(q6_benchmark)
print({'speedup': q6_speedup, 'batch_tool': q6_batch_tool, 'live_tool': q6_live_tool})

In [ ]:
reflection_rebuild = """
Hardest moment

The trickiest part was correctly separating the batch and live workflows while rebuilding the pipeline. It was tempting to reuse pandas logic inside the live validator, but that led to repeated lookups and inefficiencies. Debugging issues like missing keys and handling None values (e.g., unknown households) also made it clear that live validation requires careful conditional checks and defensive coding, unlike batch operations.

AI interaction

I used AI to help structure both the batch pipeline and the live validator, but it did not always choose the right abstraction on its own. It often suggested pandas-based solutions even for live validation tasks, which would not scale. I had to refine prompts and critically evaluate the output to ensure that batch logic used pandas, while live validation relied on dicts and sets for efficient lookups.

Surprise

The most surprising result was the magnitude of the speedup in the benchmark. Even for a relatively small sample, the dict/set-based live validator was significantly faster than the naive approach. This made it clear that inefficiencies like repeated list scans or DataFrame operations inside loops scale very poorly, and that choosing the right data structure has a large real-world impact.
"""


---

### Q7: AI Optimization Review

Two AI suggestions claim to “optimize everything.” Identify the abstraction mistake in each one and say what the deployment fix should be.


In [ ]:
q7_snippet_a = """
def ai_fast_validator(claim, households, district_rules, prior_payouts, watchlist_ids):
    watchlist_set = set(watchlist_ids)
    paid_set = set(prior_payouts['household_id'])
    district_lookup = district_rules.set_index('district').to_dict('index')
    return claim['household_id'] in watchlist_set, claim['household_id'] in paid_set, claim['district'] in district_lookup
"""

q7_snippet_b = """
def ai_universal_solution(single_claim, households, district_rules):
    claim_df = pd.DataFrame([single_claim])
    merged = claim_df.merge(households, on='household_id', how='left')
    merged = merged.merge(district_rules, left_on='district', right_on='district', how='left')
    return merged
"""

print('Snippet A\n', q7_snippet_a)
print('Snippet B\n', q7_snippet_b)

q7_assessment = {
    'snippet_a_problem': "It rebuilds sets and lookup dictionaries inside the function for every single claim, causing repeated preprocessing work and defeating the purpose of using fast lookup structures. This makes the validator inefficient in a live stream setting.",

    'snippet_b_problem': "It uses pandas merges for a single incoming claim, treating a live validation problem as a batch operation. This introduces unnecessary overhead and repeated DataFrame operations for each claim, which does not scale.",

    'best_fix': "Precompute lookup structures (dicts and sets) once and reuse them for all incoming claims in the live validator. Use pandas only for batch preprocessing, and use dict/set-based lookups for live validation to ensure O(1) operations per claim."
}

---

### Q8: Recommendation Memo

You are writing to the state deployment team. Choose the batch tool, choose the live-validation tool, and justify the split in a short memo.


In [ ]:
q8_batch_choice = "pandas"
q8_live_choice = "dict/set"

q8_memo = """
To: State Deployment Team

For the batch payout pipeline, pandas is the appropriate tool. The workflow involves full-table operations such as deduplication, merging household and district data, and computing aggregated values. These are naturally expressed as vectorized DataFrame operations and scale efficiently for large datasets.

For the live validation layer, dicts and sets should be used. Incoming claims arrive one at a time and require repeated membership checks and key-based lookups. Precomputing lookup structures allows each validation to run in constant time, avoiding repeated DataFrame scans and ensuring the system scales with high claim volumes.

The key design principle is to separate batch processing from live validation. Pandas is well-suited for one-time transformations on complete datasets, while dicts and sets are essential for fast, incremental validation in a streaming context. This split ensures both correctness and performance at deployment scale.
"""


## Q9: Safe Failure

Build a robust summary helper for validated claims that handles missing data, duplicates, and edge cases safely.


In [ ]:
import pandas as pd

def safe_topup_summary(validated_claims):
    """Return a safe summary of validated live claims."""

    # Handle empty input
    if validated_claims is None or len(validated_claims) == 0:
        return {
            'n_claims': 0,
            'n_approved': 0,
            'n_review': 0,
            'n_rejected': 0,
            'approved_total': 0.0,
            'district_counts': {}
        }

    # Convert to DataFrame if list of dicts
    if isinstance(validated_claims, list):
        df = pd.DataFrame(validated_claims)
    else:
        df = validated_claims.copy()

    # Deduplicate: keep first occurrence of claim_id
    if 'claim_id' in df.columns:
        df = df.drop_duplicates(subset='claim_id', keep='first')

    # Ensure approved_amount exists and fill missing as 0.0
    if 'approved_amount' not in df.columns:
        df['approved_amount'] = 0.0
    df['approved_amount'] = df['approved_amount'].fillna(0.0)

    # Ensure status exists
    if 'status' not in df.columns:
        df['status'] = None

    # Ensure district exists
    if 'district' not in df.columns:
        df['district'] = 'UNKNOWN'
    df['district'] = df['district'].fillna('UNKNOWN')

    # Counts
    n_claims = len(df)
    n_approved = (df['status'] == 'approve').sum()
    n_review = (df['status'] == 'review').sum()
    n_rejected = (df['status'] == 'reject').sum()

    # Sum approved_amount (include review)
    approved_total = float(df['approved_amount'].sum())

    # District counts
    district_counts = df['district'].value_counts().to_dict()

    return {
        'n_claims': int(n_claims),
        'n_approved': int(n_approved),
        'n_review': int(n_review),
        'n_rejected': int(n_rejected),
        'approved_total': approved_total,
        'district_counts': district_counts
    }

sample_live_state = build_live_state(households, district_rules, prior_payouts, watchlist_ids)
sample_results = run_fast_live_validation(list(stream_incoming_claims(limit=25)), sample_live_state, validate_incoming_claim)
q9_summary_preview = safe_topup_summary(sample_results)
print(q9_summary_preview)


In [ ]:
reflection_judge = """
Hardest moment

The hardest part was judging Snippet A fairly. It looked efficient because it used sets and dictionaries, which are the right data structures, but the abstraction was wrong because those structures were rebuilt for every claim. Recognizing that the issue was not the tool itself but where and how it was used made it harder to evaluate than Snippet B, which was more obviously inefficient.

AI interaction

I did use AI to help critique the snippets, but it initially focused on surface-level optimizations rather than the deeper abstraction issue. It correctly identified that sets and dicts are faster than lists, but it did not immediately catch that rebuilding them inside the function defeats their purpose. This reinforced the need to evaluate AI suggestions critically, especially in terms of scalability and workflow context.

Surprise

The biggest takeaway was that faster-looking code can still be the wrong solution if it is applied in the wrong context. Snippet A used “fast” data structures but still performed poorly due to repeated work, while Snippet B used powerful pandas operations in a setting where they were unnecessary. This showed that choosing the right abstraction matters more than just choosing a fast-looking tool.
"""


---
